
# Data Scientist Final Clean Notebook

## Agriculture Health AI Project

This notebook contains the Data Scientist part of the project. The main objective is to train and compare CNN transfer learning models for plant leaf disease classification.

The models used are:

- MobileNetV3Small
- ResNet50
- DenseNet121

The models were compared using:

- Test accuracy
- Mean Average Precision, mAP
- Training time
- Total parameters



## Data Scientist Role

The Data Scientist role focuses on data modelling. This includes creating CNN models, applying transfer learning, training the models, performing hyperparameter tuning, and comparing model performance.


In [1]:

import tensorflow as tf
import pandas as pd
import numpy as np
from pathlib import Path

from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Small, ResNet50, DenseNet121
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0



## Dataset Loading

The dataset was prepared by the Data Engineer and divided into training, validation, and testing folders.

The dataset contains 5 plant disease classes:

- leaf_blight
- leaf_rust
- leaf_spot
- powdery_mildew
- yellow_leaf_disease


In [2]:

# Path setup
current_dir = Path.cwd()

if current_dir.name == "Data_Scientist":
    data_scientist_dir = current_dir
    project_dir = current_dir.parent
else:
    project_dir = current_dir
    data_scientist_dir = project_dir / "Data_Scientist"

dataset_dir = project_dir / "dataset"
results_dir = data_scientist_dir / "results"

train_dir = dataset_dir / "train"
val_dir = dataset_dir / "val"
test_dir = dataset_dir / "test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Class names:", class_names)
print("Number of classes:", num_classes)


Found 4060 files belonging to 5 classes.
Found 820 files belonging to 5 classes.
Found 870 files belonging to 5 classes.
Class names: ['leaf_blight', 'leaf_rust', 'leaf_spot', 'powdery_mildew', 'yellow_leaf_disease']
Number of classes: 5



## Transfer Learning Setup

Transfer learning was used by loading pre-trained CNN models with ImageNet weights. The original classification layer was removed using `include_top=False`, and a new Dense output layer was added for the 5 plant disease classes.

The base model layers were frozen during final training to reduce training time and avoid overfitting.


In [3]:

def build_transfer_model(model_name, learning_rate=0.001, dropout_rate=0.3, fine_tune_last=0):
    inputs = tf.keras.Input(shape=(224, 224, 3))

    if model_name == "MobileNetV3Small":
        base_model = MobileNetV3Small(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3),
            pooling="avg"
        )
        x = base_model(inputs, training=False)

    elif model_name == "ResNet50":
        base_model = ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3),
            pooling="avg"
        )
        x = layers.Lambda(resnet_preprocess)(inputs)
        x = base_model(x, training=False)

    elif model_name == "DenseNet121":
        base_model = DenseNet121(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3),
            pooling="avg"
        )
        x = layers.Lambda(densenet_preprocess)(inputs)
        x = base_model(x, training=False)

    else:
        raise ValueError("Unsupported model name.")

    if fine_tune_last == 0:
        base_model.trainable = False
    else:
        base_model.trainable = True
        for layer in base_model.layers[:-fine_tune_last]:
            layer.trainable = False

    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name=model_name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("Transfer learning model function is ready.")


Transfer learning model function is ready.



## Hyperparameter Tuning

A small hyperparameter tuning experiment was conducted before final model comparison. Three trials were tested for each CNN model using different learning rates, dropout values, and trainable layer settings.

| Trial | Learning Rate | Dropout | Trainable Layers |
|---|---:|---:|---|
| Trial 1 | 0.001 | 0.3 | Frozen base |
| Trial 2 | 0.0001 | 0.4 | Frozen base |
| Trial 3 | 0.00001 | 0.3 | Fine-tune last 10 layers |

Each trial was trained for 3 epochs to compare validation accuracy.


In [4]:

# Load hyperparameter tuning result
tuning_results = pd.read_csv(results_dir / "hyperparameter_tuning_results.csv")
selected_hyperparameters = pd.read_csv(results_dir / "selected_hyperparameters.csv")

print("Hyperparameter tuning results:")
display(tuning_results)

print("Selected hyperparameters:")
display(selected_hyperparameters)


Hyperparameter tuning results:


,Model,Trial,Learning Rate,Dropout,Trainable Layers,Epochs,Best Validation Accuracy,Best Validation Accuracy (%),Final Validation Accuracy,Final Validation Accuracy (%),Final Validation Loss,Training Time (seconds),Training Time (minutes),Total Parameters
0,MobileNetV3Small,Trial 1,0.00100,0.3,Frozen base,3,0.602439,60.24,0.602439,60.24,1.0609,32.626983,0.54,942005
1,MobileNetV3Small,Trial 2,0.00010,0.4,Frozen base,3,0.367073,36.71,0.367073,36.71,1.4710,30.339514,0.51,942005
2,MobileNetV3Small,Trial 3,0.00001,0.3,Fine-tune last 10 layers,3,0.234146,23.41,0.234146,23.41,1.6953,33.746631,0.56,942005
3,ResNet50,Trial 1,0.00100,0.3,Frozen base,3,0.634146,63.41,0.634146,63.41,0.9885,350.440828,5.84,23597957
4,ResNet50,Trial 2,0.00010,0.4,Frozen base,3,0.501220,50.12,0.501220,50.12,1.2536,364.494770,6.07,23597957
5,ResNet50,Trial 3,0.00001,0.3,Fine-tune last 10 layers,3,0.531707,53.17,0.531707,53.17,1.2114,385.284122,6.42,23597957
6,DenseNet121,Trial 1,0.00100,0.3,Frozen base,3,0.584146,58.41,0.584146,58.41,1.0733,371.988369,6.20,7042629
7,DenseNet121,Trial 2,0.00010,0.4,Frozen base,3,0.339024,33.90,0.339024,33.90,1.5101,376.316512,6.27,7042629
8,DenseNet121,Trial 3,0.00001,0.3,Fine-tune last 10 layers,3,0.214634,21.46,0.214634,21.46,1.7155,385.291004,6.42,7042629


Selected hyperparameters:


,Model,Trial,Learning Rate,Dropout,Trainable Layers,Epochs,Best Validation Accuracy (%),Final Validation Accuracy (%),Training Time (minutes)
0,DenseNet121,Trial 1,0.001,0.3,Frozen base,3,58.41,58.41,6.20
1,MobileNetV3Small,Trial 1,0.001,0.3,Frozen base,3,60.24,60.24,0.54
2,ResNet50,Trial 1,0.001,0.3,Frozen base,3,63.41,63.41,5.84



## Final Training Configuration

Based on the hyperparameter tuning results, Trial 1 was selected for final training:

- Learning rate: 0.001
- Dropout: 0.3
- Trainable layers: Frozen base
- Epochs: 50

The final models were trained for 50 epochs each.


In [5]:

# Final training code reference
# This cell is kept as documentation.
# Do not run this cell unless you want to retrain the models again.

RUN_FINAL_TRAINING = False

if RUN_FINAL_TRAINING:
    final_models = ["MobileNetV3Small", "ResNet50", "DenseNet121"]

    for model_name in final_models:
        print("Training:", model_name)

        model = build_transfer_model(
            model_name=model_name,
            learning_rate=0.001,
            dropout_rate=0.3,
            fine_tune_last=0
        )

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=50
        )
else:
    print("Final training was already completed in the working notebook.")
    print("Saved result CSV files are used for final comparison.")


Final training was already completed in the working notebook.
Saved result CSV files are used for final comparison.



## Final Model Comparison

The final trained models were evaluated using the testing dataset. The comparison considers test accuracy, mAP, training time, and total parameters.


In [6]:

final_results = pd.read_csv(results_dir / "final_model_comparison_rounded.csv")
display(final_results)


,Model,Test Accuracy,Test Accuracy (%),Test Loss,mAP,mAP (%),Training Time (seconds),Training Time (minutes),Total Parameters
0,ResNet50,0.7115,71.15,0.8988,0.7856,78.56,5880.30,98.01,23597957
1,DenseNet121,0.6356,63.56,0.9478,0.7058,70.58,5988.47,99.81,7042629
2,MobileNetV3Small,0.6264,62.64,0.9785,0.6899,68.99,492.15,8.20,942005



## Final Conclusion

Based on the final comparison, ResNet50 achieved the best performance with the highest test accuracy and mAP. Therefore, ResNet50 is selected as the best model for plant leaf disease classification performance.

MobileNetV3Small was the fastest and lightest model, making it more suitable for lightweight applications. However, its accuracy and mAP were lower compared to ResNet50.
